In [1]:
SLICE_NAME = 'qfabric-bb84-2'
SCENARIO   = 'validation/scenarios/fabric_1km.yml'

In [2]:
import os, sys, json
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR)); sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
import deploy_fabric_modified as df

fablib = df.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid


ID,d8ea8804-1e65-4186-a902-d6006de32667
Name,qfabric-bb84-2
Lease Expiration (UTC),2026-08-04 21:27:56 +0000
Lease Start (UTC),2026-07-20 21:27:56 +0000
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
State,StableOK
Email,audreyf@illinois.edu
UserId,8616dd8c-5b61-45db-84bb-791c21e82a89


ID,d8ea8804-1e65-4186-a902-d6006de32667
Name,qfabric-bb84-2
Lease Expiration (UTC),2026-08-04 21:27:56 +0000
Lease Start (UTC),2026-07-20 21:27:56 +0000
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
State,StableOK
Email,audreyf@illinois.edu
UserId,8616dd8c-5b61-45db-84bb-791c21e82a89


In [9]:
from scipy.stats import qmc
import pandas as pd

# 6 dimensions: loss, latency, jitter, polarization fidelity, attenuation, distance
sampler = qmc.LatinHypercube(d=6, seed=42)
n_points = 40
sample = sampler.random(n=n_points)

bounds_lower = [0,    10,  0,   0.8, 0.1, 1]     # loss%, latency_ms, jitter_ms, pf, atten_db_per_km, dist_km
bounds_upper = [15,  200, 30,   1.0, 0.5, 100]

scaled_sample = qmc.scale(sample, bounds_lower, bounds_upper)

df_lhs_plan = pd.DataFrame(scaled_sample, columns=[
    'loss_pct', 'latency_ms', 'jitter_ms',
    'polarization_fidelity', 'attenuation_db_per_km', 'distance_km'
])

print(df_lhs_plan.head(10))
print(f"\nTotal points planned: {len(df_lhs_plan)}")

    loss_pct  latency_ms  jitter_ms  polarization_fidelity  \
0   3.459766   26.915327  18.856052               0.836513   
1   1.589573  148.766195  28.403915               0.857748   
2  11.008551   67.841882   7.167439               0.988864   
3   3.814638   30.749594   6.181434               0.878227   
4  12.083106  104.075466  25.149959               0.899781   
5  11.720714  124.154329   0.505631               0.803148   
6   8.951279   12.490402   5.079818               0.866651   
7  12.862401   89.266258  16.625805               0.815976   
8  13.994064  161.336176  21.600069               0.884963   
9   7.235563   20.541537  17.655813               0.912156   

   attenuation_db_per_km  distance_km  
0               0.389058    90.160335  
1               0.396292    28.406257  
2               0.144454    52.817052  
3               0.150293    65.614525  
4               0.378457    31.484454  
5               0.175304    77.256058  
6               0.225628    68.239121

In [10]:
import os, csv, json
import deploy_fabric_modified as deploy

FIELDNAMES_LHS = ['condition', 'loss_pct', 'delay_ms', 'jitter_ms', 'qber', 'sifted_bits',
                    'final_key_bits', 'secure_key_rate', 'elapsed_seconds', 'key_bits_per_sec',
                    'polarization_fidelity', 'attenuation_db_per_km', 'distance_km', 'lhs_point_id', 'run', 'note']

def run_lhs_sweep(slice_obj, df_lhs_plan, n_runs=1,
                    out_csv='/home/fabric/work/qkd-pqc-dependability/results/joint_fault_lhs_results.csv'):
    all_rows = []
    total = len(df_lhs_plan) * n_runs
    done = 0
    write_header = not os.path.exists(out_csv)

    for idx, point in df_lhs_plan.iterrows():
        pf = round(point['polarization_fidelity'], 3)
        atten = round(point['attenuation_db_per_km'], 3)
        dist = round(point['distance_km'], 1)
        loss = round(point['loss_pct'], 2)
        delay = round(point['latency_ms'], 1)
        jitter = round(point['jitter_ms'], 1)

        scenario_content = f"""name: lhs_{idx}
channel:
  distance_km: {dist}
  attenuation_db_per_km: {atten}
  polarization_fidelity: {pf}
detector:
  efficiency: 0.8
  dark_count_rate: 10.0
  dead_time: 0.0
  timing_jitter: 0.0
protocol:
  num_photons: 10000
  send_rate_hz: 10000.0
  sample_fraction: 0.1
  wavelength: 0
seed: 42
"""
        scenario_path = f'/home/fabric/work/qkd-pqc-dependability/validation/scenarios/temp_lhs_{idx}.yml'
        with open(scenario_path, 'w') as f:
            f.write(scenario_content)

        condition = [{'name': f'lhs_{idx}', 'loss_pct': loss, 'delay_ms': delay, 'jitter_ms': jitter}]

        for run in range(n_runs):
            rows = deploy.run_network_conditions_experiment(slice_obj, scenario_path, conditions=condition, save_network_effects_json=False)
            for row in rows:
                row['polarization_fidelity'] = pf
                row['attenuation_db_per_km'] = atten
                row['distance_km'] = dist
                row['lhs_point_id'] = idx
                row['run'] = run + 1
                all_rows.append(row)
                with open(out_csv, 'a', newline='') as f:
                    writer = csv.DictWriter(f, fieldnames=FIELDNAMES_LHS, extrasaction='ignore')
                    if write_header:
                        writer.writeheader()
                        write_header = False
                    writer.writerow(row)
            done += 1
            print(f"Progress: {done}/{total} runs complete (LHS point {idx})")

    with open('/home/fabric/work/qkd-pqc-dependability/results/joint_fault_lhs_results.json', 'w') as f:
        json.dump(all_rows, f, indent=2)
    return all_rows

In [12]:
rows_lhs = run_lhs_sweep(slice_obj, df_lhs_plan, n_runs=1)


##### [1/1] classical condition: lhs_0 #####
  cleared classical netem on Alice and Bob

=== Applying classical-channel netem (TCP:5100) ===
  alice (enp7s0): netem delay 26.9ms 18.9ms loss 3.46%
  bob (enp7s0): netem delay 26.9ms 18.9ms loss 3.46%

=== Running BB84 protocol ===
  Bob data-plane IP: 10.10.1.2 (for classical channel)
  Alice data iface:  enp7s0
  Bob data iface:    enp7s0
  Cleaning up previous runs...
  Ensuring deps on alice...
  Ensuring deps on bob...
  Starting Bob...
  Alice MAC args:  --dst-mac '1A:9E:9B:43:74:D7' --src-mac '0E:A9:B5:58:70:AB'
  Starting Alice...
  Waiting for BB84 to complete...
  Alice finished
  Alice output: Alice: Sending 10000 photons on enp7s0
Alice: Finished sending 10000 photons
Alice: Connecting to Bob at 10.10.1.2:5100
  Retrying connection to 10.10.1.2:5100 (attempt 2/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 3/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 4/60)...
  Retrying connection to 10.10.1.2:5100 (atte

In [13]:
import pandas as pd

df_lhs = pd.read_csv('/home/fabric/work/qkd-pqc-dependability/results/joint_fault_lhs_results.csv')
print(f"Total rows: {len(df_lhs)}")
print(df_lhs[['lhs_point_id']].nunique())

for c in ['qber', 'sifted_bits', 'key_bits_per_sec', 'elapsed_seconds']:
    df_lhs[c] = pd.to_numeric(df_lhs[c], errors='coerce')

# Plausibility check: sifted_bits should track num_photons=10000 regardless of parameters
suspect = df_lhs[(df_lhs['sifted_bits'] < 3000) | (df_lhs['sifted_bits'] > 4200)]
print(f"\nSuspect rows (contamination or unusual physics): {len(suspect)}")
print(suspect[['lhs_point_id', 'polarization_fidelity', 'attenuation_db_per_km',
                'distance_km', 'loss_pct', 'sifted_bits', 'qber']])

# Check for the duplicate-row contamination signature
dupes = df_lhs[df_lhs.duplicated(subset=['elapsed_seconds', 'sifted_bits'], keep=False)]
print(f"\nSuspicious duplicate rows: {len(dupes)}")

# How many points actually reached/crossed your failure threshold?
failures = df_lhs[(df_lhs['qber'] >= 0.11) | (df_lhs['key_bits_per_sec'] <= 0)]
print(f"\nFailure points found: {len(failures)}")
print(failures[['lhs_point_id', 'polarization_fidelity', 'attenuation_db_per_km',
                 'distance_km', 'loss_pct', 'delay_ms', 'qber', 'key_bits_per_sec']])

Total rows: 40
lhs_point_id    40
dtype: int64

Suspect rows (contamination or unusual physics): 0
Empty DataFrame
Columns: [lhs_point_id, polarization_fidelity, attenuation_db_per_km, distance_km, loss_pct, sifted_bits, qber]
Index: []

Suspicious duplicate rows: 0

Failure points found: 0
Empty DataFrame
Columns: [lhs_point_id, polarization_fidelity, attenuation_db_per_km, distance_km, loss_pct, delay_ms, qber, key_bits_per_sec]
Index: []


In [14]:
print(sorted(df_lhs['polarization_fidelity'].round(3).tolist()))

[0.803, 0.805, 0.81, 0.816, 0.822, 0.828, 0.835, 0.837, 0.841, 0.848, 0.854, 0.858, 0.864, 0.867, 0.87, 0.878, 0.885, 0.888, 0.893, 0.9, 0.903, 0.908, 0.912, 0.916, 0.923, 0.928, 0.931, 0.935, 0.944, 0.947, 0.951, 0.958, 0.962, 0.969, 0.972, 0.976, 0.983, 0.989, 0.991, 0.997]


In [15]:
df_lhs['dist_to_threshold'] = (df_lhs['qber'] - 0.11).abs()
print(df_lhs.sort_values('dist_to_threshold')[['lhs_point_id', 'polarization_fidelity',
      'attenuation_db_per_km', 'loss_pct', 'qber', 'dist_to_threshold']].head(5))

    lhs_point_id  polarization_fidelity  attenuation_db_per_km  loss_pct  \
5              5                  0.803                  0.175     11.72   
21            21                  0.822                  0.368      1.00   
7              7                  0.816                  0.356     12.86   
0              0                  0.837                  0.389      3.46   
16            16                  0.805                  0.241      9.61   

        qber  dist_to_threshold  
5   0.107050           0.002950  
21  0.093834           0.016166  
7   0.093583           0.016417  
0   0.091384           0.018616  
16  0.091153           0.018847  


In [16]:
row5 = df_lhs[df_lhs['lhs_point_id'] == 5].iloc[0]
print(row5)

condition                     lhs_5
loss_pct                      11.72
delay_ms                      124.2
jitter_ms                       0.5
qber                        0.10705
sifted_bits                    3837
final_key_bits                   62
secure_key_rate              0.0062
elapsed_seconds          103.764999
key_bits_per_sec           0.597504
polarization_fidelity         0.803
attenuation_db_per_km         0.175
distance_km                    77.3
lhs_point_id                      5
run                               1
note                            NaN
dist_to_threshold           0.00295
Name: 5, dtype: object


In [17]:
import os, csv, json

FIELDNAMES_LHS5 = ['condition', 'loss_pct', 'delay_ms', 'jitter_ms', 'qber', 'sifted_bits',
                    'final_key_bits', 'secure_key_rate', 'elapsed_seconds', 'key_bits_per_sec',
                    'polarization_fidelity', 'attenuation_db_per_km', 'distance_km', 'run', 'note']

def run_lhs5_replicate(slice_obj, n_runs=4, start_run=2,
                         out_csv='/home/fabric/work/qkd-pqc-dependability/results/joint_fault_lhs5_replicate.csv'):
    pf = 0.803
    atten = 0.175
    dist = 77.3
    loss = 11.72
    delay = 124.2
    jitter = 0.5

    scenario_content = f"""name: lhs5_replicate
channel:
  distance_km: {dist}
  attenuation_db_per_km: {atten}
  polarization_fidelity: {pf}
detector:
  efficiency: 0.8
  dark_count_rate: 10.0
  dead_time: 0.0
  timing_jitter: 0.0
protocol:
  num_photons: 10000
  send_rate_hz: 10000.0
  sample_fraction: 0.1
  wavelength: 0
seed: 42
"""
    scenario_path = f'/home/fabric/work/qkd-pqc-dependability/validation/scenarios/temp_lhs5_replicate.yml'
    with open(scenario_path, 'w') as f:
        f.write(scenario_content)

    condition = [{'name': 'lhs5_replicate', 'loss_pct': loss, 'delay_ms': delay, 'jitter_ms': jitter}]

    all_rows = []
    write_header = not os.path.exists(out_csv)

    for run in range(n_runs):
        rows = deploy.run_network_conditions_experiment(slice_obj, scenario_path, conditions=condition,
                                                           save_network_effects_json=False)
        for row in rows:
            row['polarization_fidelity'] = pf
            row['attenuation_db_per_km'] = atten
            row['distance_km'] = dist
            row['run'] = start_run + run
            all_rows.append(row)
            with open(out_csv, 'a', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=FIELDNAMES_LHS5, extrasaction='ignore')
                if write_header:
                    writer.writeheader()
                    write_header = False
                writer.writerow(row)
        print(f"Run {start_run+run}: QBER={rows[0].get('qber')}, "
              f"crossed threshold: {rows[0].get('qber', 0) >= 0.11}")

    return all_rows

rows_lhs5 = run_lhs5_replicate(slice_obj, n_runs=4, start_run=2)


##### [1/1] classical condition: lhs5_replicate #####
  cleared classical netem on Alice and Bob

=== Applying classical-channel netem (TCP:5100) ===
  alice (enp7s0): netem delay 124.2ms 0.5ms loss 11.72%
  bob (enp7s0): netem delay 124.2ms 0.5ms loss 11.72%

=== Running BB84 protocol ===
  Bob data-plane IP: 10.10.1.2 (for classical channel)
  Alice data iface:  enp7s0
  Bob data iface:    enp7s0
  Cleaning up previous runs...
  Ensuring deps on alice...
  Ensuring deps on bob...
  Starting Bob...
  Alice MAC args:  --dst-mac '1A:9E:9B:43:74:D7' --src-mac '0E:A9:B5:58:70:AB'
  Starting Alice...
  Waiting for BB84 to complete...
  Alice finished
  Alice output: Alice: Sending 10000 photons on enp7s0
Alice: Finished sending 10000 photons
Alice: Connecting to Bob at 10.10.1.2:5100
  Retrying connection to 10.10.1.2:5100 (attempt 2/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 3/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 4/60)...
  Retrying connection to 10.10.1.2

In [18]:
import pandas as pd
df_lhs5 = pd.read_csv('/home/fabric/work/qkd-pqc-dependability/results/joint_fault_lhs5_replicate.csv')
original = pd.DataFrame([{'qber': 0.10705, 'run': 1}])
combined = pd.concat([original, df_lhs5[['qber', 'run']]], ignore_index=True)
print(combined)
print(f"\nCrossed 11% threshold in {(combined['qber'] >= 0.11).sum()} of {len(combined)} runs")

       qber  run
0  0.107050    1
1  0.091603    2
2  0.103175    3
3  0.092784    4
4  0.079897    5

Crossed 11% threshold in 0 of 5 runs


In [19]:
import pandas as pd

# Combine everything you have into one pool
df_pf = pd.read_csv('/home/fabric/work/qkd-pqc-dependability/results/joint_fault_pf_results.csv')
df_eff = pd.read_csv('/home/fabric/work/qkd-pqc-dependability/results/joint_fault_efficiency_loss.csv')
df_lhs = pd.read_csv('/home/fabric/work/qkd-pqc-dependability/results/joint_fault_lhs_results.csv')
df_lhs5 = pd.read_csv('/home/fabric/work/qkd-pqc-dependability/results/joint_fault_lhs5_replicate.csv')

for df in [df_pf, df_eff, df_lhs, df_lhs5]:
    df['qber'] = pd.to_numeric(df['qber'], errors='coerce')
    if 'key_bits_per_sec' in df.columns:
        df['key_bits_per_sec'] = pd.to_numeric(df['key_bits_per_sec'], errors='coerce')

total_failures = (
    (df_pf['qber'] >= 0.11).sum() +
    (df_eff['qber'] >= 0.11).sum() +
    (df_lhs['qber'] >= 0.11).sum() +
    (df_lhs5['qber'] >= 0.11).sum()
)
total_rows = len(df_pf) + len(df_eff) + len(df_lhs) + len(df_lhs5)
print(f"Total failures across everything: {total_failures} / {total_rows} rows")

Total failures across everything: 28 / 180 rows


In [20]:
import pandas as pd

# Tag each dataset by source and combine
df_pf['source'] = 'pf_loss'
df_eff['source'] = 'eff_loss'
df_lhs['source'] = 'lhs'
df_lhs5['source'] = 'lhs5'

df_pf['dark_count_rate'] = 10.0
df_pf['detector_efficiency'] = 0.8
df_pf['attenuation_db_per_km'] = 0.2
df_pf['distance_km'] = 10.0

df_eff['polarization_fidelity'] = 0.8
df_eff['dark_count_rate'] = 10.0
df_eff['attenuation_db_per_km'] = 0.2
df_eff['distance_km'] = 10.0

df_lhs['dark_count_rate'] = 10.0
df_lhs['detector_efficiency'] = 0.8

df_lhs5['dark_count_rate'] = 10.0
df_lhs5['detector_efficiency'] = 0.8
df_lhs5['loss_pct'] = 11.72
df_lhs5['delay_ms'] = 124.2

common_cols = ['polarization_fidelity', 'attenuation_db_per_km', 'distance_km',
               'detector_efficiency', 'dark_count_rate', 'loss_pct', 'qber', 'key_bits_per_sec', 'source']

df_all = pd.concat([df_pf[common_cols], df_eff[common_cols],
                     df_lhs[common_cols], df_lhs5[common_cols]], ignore_index=True)

df_all['qber'] = pd.to_numeric(df_all['qber'], errors='coerce')
df_all['key_bits_per_sec'] = pd.to_numeric(df_all['key_bits_per_sec'], errors='coerce')
df_all['Y'] = ((df_all['qber'] >= 0.11) | (df_all['key_bits_per_sec'] <= 0)).astype(int)

print(df_all.groupby('source')['Y'].sum())
print(f"\nTotal: {df_all['Y'].sum()} / {len(df_all)}")

failures_by_source = df_all[df_all['Y']==1]
print(failures_by_source[['source', 'polarization_fidelity', 'attenuation_db_per_km', 'loss_pct', 'qber', 'key_bits_per_sec']])

source
eff_loss    24
lhs          0
lhs5         0
pf_loss      4
Name: Y, dtype: int64

Total: 28 / 180
       source  polarization_fidelity  attenuation_db_per_km  loss_pct  \
3     pf_loss                    0.8                    0.2      15.0   
10    pf_loss                    0.8                    0.2       5.0   
36    pf_loss                    0.8                    0.2       NaN   
48    pf_loss                    0.8                    0.2       NaN   
52   eff_loss                    0.8                    0.2       NaN   
54   eff_loss                    0.8                    0.2       5.0   
55   eff_loss                    0.8                    0.2      15.0   
57   eff_loss                    0.8                    0.2       1.0   
58   eff_loss                    0.8                    0.2       5.0   
60   eff_loss                    0.8                    0.2       NaN   
66   eff_loss                    0.8                    0.2       5.0   
70   eff_loss     

In [4]:
# Quick single-point sanity check: does netem actually apply a tiny loss value?
import deploy_fabric_modified as deploy
alice = slice_obj.get_node("alice")
alice_iface = alice.get_interface(network_name="net_alice_switch").get_device_name()
deploy.clear_classical_netem(slice_obj)
deploy.apply_classical_netem(slice_obj, loss_pct=0.001)
print(alice.execute(f"sudo tc qdisc show dev {alice_iface}", quiet=False))

  cleared classical netem on Alice and Bob

=== Applying classical-channel netem (TCP:5100) ===
  alice (enp7s0): netem loss 0.001%
  bob (enp7s0): netem loss 0.001%
qdisc prio 1: root refcnt 33 bands 3 priomap 1 2 2 2 1 2 0 0 1 1 1 1 1 1 1 1
qdisc netem 30: parent 1:3 limit 1000 loss 0.00100001%
('qdisc prio 1: root refcnt 33 bands 3 priomap 1 2 2 2 1 2 0 0 1 1 1 1 1 1 1 1\nqdisc netem 30: parent 1:3 limit 1000 loss 0.00100001%\n', '')


In [5]:
import os, csv, json

FIELDNAMES_LOWLOSS = ['condition', 'loss_pct', 'qber', 'sifted_bits', 'final_key_bits',
                        'secure_key_rate', 'elapsed_seconds', 'key_bits_per_sec',
                        'polarization_fidelity', 'run', 'note']

def run_lowloss_sweep(slice_obj, pf_values, loss_values, n_runs=3,
                        out_csv='/home/fabric/work/qkd-pqc-dependability/results/joint_fault_lowloss_results.csv'):
    all_rows = []
    total = len(pf_values) * len(loss_values) * n_runs
    done = 0
    write_header = not os.path.exists(out_csv)

    for pf in pf_values:
        scenario_content = f"""name: lowloss_pf_{pf}
channel:
  distance_km: 10.0
  attenuation_db_per_km: 0.2
  polarization_fidelity: {pf}
detector:
  efficiency: 0.8
  dark_count_rate: 10.0
  dead_time: 0.0
  timing_jitter: 0.0
protocol:
  num_photons: 10000
  send_rate_hz: 10000.0
  sample_fraction: 0.1
  wavelength: 0
seed: 42
"""
        scenario_path = f'/home/fabric/work/qkd-pqc-dependability/validation/scenarios/temp_lowloss_pf_{pf}.yml'
        with open(scenario_path, 'w') as f:
            f.write(scenario_content)

        for loss in loss_values:
            condition = [{'name': f'loss_{loss}pct', 'loss_pct': loss}] if loss > 0 else [{'name': 'baseline'}]

            for run in range(n_runs):
                rows = deploy.run_network_conditions_experiment(slice_obj, scenario_path, conditions=condition,
                                                                   save_network_effects_json=False)
                for row in rows:
                    row['polarization_fidelity'] = pf
                    row['run'] = run + 1
                    all_rows.append(row)
                    with open(out_csv, 'a', newline='') as f:
                        writer = csv.DictWriter(f, fieldnames=FIELDNAMES_LOWLOSS, extrasaction='ignore')
                        if write_header:
                            writer.writeheader()
                            write_header = False
                        writer.writerow(row)
                done += 1
                print(f"Progress: {done}/{total} (pf={pf}, loss={loss}%, run={run+1})")

    with open('/home/fabric/work/qkd-pqc-dependability/results/joint_fault_lowloss_results.json', 'w') as f:
        json.dump(all_rows, f, indent=2)
    return all_rows

# Fine-grained low end, connecting up to your existing 1% data point
loss_values = [0, 0.001, 0.01, 0.1, 0.5, 1.0]
pf_values = [0.8, 1.0]  # your known boundary case + your known-safe case

rows_lowloss = run_lowloss_sweep(slice_obj, pf_values, loss_values, n_runs=3)


##### [1/1] classical condition: baseline #####
  cleared classical netem on Alice and Bob

=== Running BB84 protocol ===
  Bob data-plane IP: 10.10.1.2 (for classical channel)
  Alice data iface:  enp7s0
  Bob data iface:    enp7s0
  Cleaning up previous runs...
  Ensuring deps on alice...
  Ensuring deps on bob...
  Starting Bob...
  Alice MAC args:  --dst-mac '1A:9E:9B:43:74:D7' --src-mac '0E:A9:B5:58:70:AB'
  Starting Alice...
  Waiting for BB84 to complete...
  Alice finished
  Alice output: Alice: Sending 10000 photons on enp7s0
Alice: Finished sending 10000 photons
Alice: Connecting to Bob at 10.10.1.2:5100
  Retrying connection to 10.10.1.2:5100 (attempt 2/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 3/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 4/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 5/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 6/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 7/60)...
  Retrying connection to 10.1

In [6]:
import pandas as pd

df_lowloss = pd.read_csv('/home/fabric/work/qkd-pqc-dependability/results/joint_fault_lowloss_results.csv')
print(f"Total rows: {len(df_lowloss)}")
print(df_lowloss.groupby(['polarization_fidelity', 'condition']).size())

for c in ['qber', 'sifted_bits', 'key_bits_per_sec', 'elapsed_seconds']:
    df_lowloss[c] = pd.to_numeric(df_lowloss[c], errors='coerce')

# Plausibility check
suspect = df_lowloss[(df_lowloss['sifted_bits'] < 3000) | (df_lowloss['sifted_bits'] > 4200)]
print(f"\nSuspect rows: {len(suspect)}")

# Check for duplicate contamination
dupes = df_lowloss[df_lowloss.duplicated(subset=['elapsed_seconds', 'sifted_bits'], keep=False)]
print(f"Suspicious duplicates: {len(dupes)}")

Total rows: 36
polarization_fidelity  condition    
0.8                    baseline         3
                       loss_0.001pct    3
                       loss_0.01pct     3
                       loss_0.1pct      3
                       loss_0.5pct      3
                       loss_1.0pct      3
1.0                    baseline         3
                       loss_0.001pct    3
                       loss_0.01pct     3
                       loss_0.1pct      3
                       loss_0.5pct      3
                       loss_1.0pct      3
dtype: int64

Suspect rows: 0
Suspicious duplicates: 0


In [7]:
df_lowloss['loss_pct'] = df_lowloss['loss_pct'].fillna(0)

summary = df_lowloss.groupby(['polarization_fidelity', 'loss_pct']).agg(
    qber_mean=('qber', 'mean'),
    elapsed_mean=('elapsed_seconds', 'mean'),
    keyrate_mean=('key_bits_per_sec', 'mean'),
).reset_index()

print(summary.to_string())

    polarization_fidelity  loss_pct  qber_mean  elapsed_mean  keyrate_mean
0                     0.8     0.000   0.104003     49.214493      6.778147
1                     0.8     0.001   0.105104     49.096535      4.122037
2                     0.8     0.010   0.104802     48.785334      4.344134
3                     0.8     0.100   0.087507     48.817661      9.985723
4                     0.8     0.500   0.094789     50.605051      6.432468
5                     0.8     1.000   0.096184     49.702963      6.066026
6                     1.0     0.000   0.000000     52.088485     66.196055
7                     1.0     0.001   0.000000     48.591669     69.956377
8                     1.0     0.010   0.000000     49.014441     69.965011
9                     1.0     0.100   0.000000     50.020538     68.971472
10                    1.0     0.500   0.000000     49.119896     70.313070
11                    1.0     1.000   0.000000     50.535037     67.674024


In [8]:
loss_values_gap = [2, 3]
pf_values = [0.8, 1.0]

rows_gap = run_lowloss_sweep(slice_obj, pf_values, loss_values_gap, n_runs=3,
                               out_csv='/home/fabric/work/qkd-pqc-dependability/results/joint_fault_lowloss_results.csv')


##### [1/1] classical condition: loss_2pct #####
  cleared classical netem on Alice and Bob

=== Applying classical-channel netem (TCP:5100) ===
  alice (enp7s0): netem loss 2%
  bob (enp7s0): netem loss 2%

=== Running BB84 protocol ===
  Bob data-plane IP: 10.10.1.2 (for classical channel)
  Alice data iface:  enp7s0
  Bob data iface:    enp7s0
  Cleaning up previous runs...
  Ensuring deps on alice...
  Ensuring deps on bob...
  Starting Bob...
  Alice MAC args:  --dst-mac '1A:9E:9B:43:74:D7' --src-mac '0E:A9:B5:58:70:AB'
  Starting Alice...
  Waiting for BB84 to complete...
  Alice finished
  Alice output: Alice: Sending 10000 photons on enp7s0
Alice: Finished sending 10000 photons
Alice: Connecting to Bob at 10.10.1.2:5100
  Retrying connection to 10.10.1.2:5100 (attempt 2/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 3/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 4/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 5/60)...
  Retrying connection to 10.1

In [9]:
import pandas as pd

df_lowloss = pd.read_csv('/home/fabric/work/qkd-pqc-dependability/results/joint_fault_lowloss_results.csv')
df_lowloss['loss_pct'] = df_lowloss['loss_pct'].fillna(0)
for c in ['qber', 'sifted_bits', 'key_bits_per_sec', 'elapsed_seconds']:
    df_lowloss[c] = pd.to_numeric(df_lowloss[c], errors='coerce')

summary = df_lowloss.groupby(['polarization_fidelity', 'loss_pct']).agg(
    qber_mean=('qber', 'mean'),
    elapsed_mean=('elapsed_seconds', 'mean'),
    keyrate_mean=('key_bits_per_sec', 'mean'),
).reset_index()

print(summary.to_string())

    polarization_fidelity  loss_pct  qber_mean  elapsed_mean  keyrate_mean
0                     0.8     0.000   0.104003     49.214493      6.778147
1                     0.8     0.001   0.105104     49.096535      4.122037
2                     0.8     0.010   0.104802     48.785334      4.344134
3                     0.8     0.100   0.087507     48.817661      9.985723
4                     0.8     0.500   0.094789     50.605051      6.432468
5                     0.8     1.000   0.096184     49.702963      6.066026
6                     0.8     2.000   0.102627     52.570815      3.502169
7                     0.8     3.000   0.100338     50.962317      4.091064
8                     1.0     0.000   0.000000     52.088485     66.196055
9                     1.0     0.001   0.000000     48.591669     69.956377
10                    1.0     0.010   0.000000     49.014441     69.965011
11                    1.0     0.100   0.000000     50.020538     68.971472
12                    1.0

In [13]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF

df1 = pd.read_csv('/home/fabric/work/qkd-pqc-dependability/results/joint_fault_pf_results.csv')
df2 = pd.read_csv('/home/fabric/work/qkd-pqc-dependability/results/joint_fault_lowloss_results.csv')

for df in [df1, df2]:
    df['loss_pct'] = df['loss_pct'].fillna(0)
    df['qber'] = pd.to_numeric(df['qber'], errors='coerce')
    df['key_bits_per_sec'] = pd.to_numeric(df['key_bits_per_sec'], errors='coerce')

df_combined = pd.concat([df1[['polarization_fidelity','loss_pct','qber','key_bits_per_sec']],
                          df2[['polarization_fidelity','loss_pct','qber','key_bits_per_sec']]], ignore_index=True)
df_combined['Y'] = ((df_combined['qber'] >= 0.11) | (df_combined['key_bits_per_sec'] <= 0)).astype(int)
print(df_combined['Y'].value_counts())

X_pool = df_combined[['polarization_fidelity', 'loss_pct']].values.astype(float)
y_pool = df_combined['Y'].values.astype(float)

X_mean, X_std = X_pool.mean(axis=0), X_pool.std(axis=0)
X_norm = (X_pool - X_mean) / X_std

kernel = RBF(length_scale=[1.0, 1.0])
gp = GaussianProcessClassifier(kernel=kernel)
gp.fit(X_norm, y_pool)

pf_grid = np.linspace(0.80, 1.0, 30)
loss_grid = np.linspace(0, 15, 30)
candidates = np.array([[pf, l] for pf in pf_grid for l in loss_grid])
candidates_norm = (candidates - X_mean) / X_std

probs = gp.predict_proba(candidates_norm)[:, 1]
uncertainty = np.abs(probs - 0.5)
top5_idx = np.argsort(uncertainty)[:5]

print("\nTop 5 model-recommended next queries:")
for i in top5_idx:
    print(f"  pf={candidates[i,0]:.3f}, loss={candidates[i,1]:.2f}, P(fail)={probs[i]:.3f}")

Y
0    90
1    10
Name: count, dtype: int64

Top 5 model-recommended next queries:
  pf=0.800, loss=15.00, P(fail)=0.187
  pf=0.800, loss=14.48, P(fail)=0.187
  pf=0.800, loss=13.97, P(fail)=0.187
  pf=0.800, loss=13.45, P(fail)=0.187
  pf=0.800, loss=12.93, P(fail)=0.187


In [14]:
print(gp.kernel_)
print(f"Max P(fail) in grid: {probs.max():.3f}")
print(f"Min P(fail) in grid: {probs.min():.3f}")

RBF(length_scale=[1.94, 1.03e+04])
Max P(fail) in grid: 0.187
Min P(fail) in grid: 0.067


In [23]:
# Quick single-point check: does jitter, combined with your known marginal PF, do anything at all?
def run_jitter_check(slice_obj, pf=0.8, jitter_values=[10, 30, 50], n_runs=2,
                       out_csv='/home/fabric/work/qkd-pqc-dependability/results/joint_fault_jitter_check.csv'):
    scenario_content = f"""name: jitter_check_pf{pf}
channel:
  distance_km: 10.0
  attenuation_db_per_km: 0.2
  polarization_fidelity: {pf}
detector:
  efficiency: 0.8
  dark_count_rate: 10.0
  dead_time: 0.0
  timing_jitter: 0.0
protocol:
  num_photons: 10000
  send_rate_hz: 10000.0
  sample_fraction: 0.1
  wavelength: 0
seed: 42
"""
    scenario_path = f'/home/fabric/work/qkd-pqc-dependability/validation/scenarios/temp_jitter_check.yml'
    with open(scenario_path, 'w') as f:
        f.write(scenario_content)

    import os, csv
    fieldnames = ['condition', 'delay_ms', 'jitter_ms', 'qber', 'sifted_bits', 'final_key_bits',
                  'secure_key_rate', 'elapsed_seconds', 'key_bits_per_sec', 'run', 'note']
    write_header = not os.path.exists(out_csv)

    for jitter in jitter_values:
        condition = [{'name': f'jitter_{jitter}ms', 'delay_ms': 50, 'jitter_ms': jitter}]
        for run in range(n_runs):
            rows = deploy.run_network_conditions_experiment(slice_obj, scenario_path, conditions=condition,
                                                               save_network_effects_json=False)
            for row in rows:
                row['run'] = run + 1
                with open(out_csv, 'a', newline='') as f:
                    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
                    if write_header:
                        writer.writeheader()
                        write_header = False
                    writer.writerow(row)
            print(f"jitter={jitter}ms, run={run+1}: QBER={rows[0].get('qber')}")

run_jitter_check(slice_obj)


##### [1/1] classical condition: jitter_10ms #####
  cleared classical netem on Alice and Bob

=== Applying classical-channel netem (TCP:5100) ===
  alice (enp7s0): netem delay 50ms 10ms
  bob (enp7s0): netem delay 50ms 10ms

=== Running BB84 protocol ===
  Bob data-plane IP: 10.10.1.2 (for classical channel)
  Alice data iface:  enp7s0
  Bob data iface:    enp7s0
  Cleaning up previous runs...
  Ensuring deps on alice...
  Ensuring deps on bob...
  Starting Bob...
  Alice MAC args:  --dst-mac '1A:9E:9B:43:74:D7' --src-mac '0E:A9:B5:58:70:AB'
  Starting Alice...
  Waiting for BB84 to complete...
  Alice finished
  Alice output: Alice: Sending 10000 photons on enp7s0
Alice: Finished sending 10000 photons
Alice: Connecting to Bob at 10.10.1.2:5100
  Retrying connection to 10.10.1.2:5100 (attempt 2/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 3/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 4/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 5/60)...
  Retrying 

In [3]:
import os, csv
import deploy_fabric_modified as deploy

FIELDNAMES_DIST2 = ['condition', 'loss_pct', 'qber', 'sifted_bits', 'final_key_bits',
                      'secure_key_rate', 'elapsed_seconds', 'key_bits_per_sec',
                      'distance_km', 'run', 'note']

def run_dist_loss_v2(slice_obj, dist_values, conditions, n_runs=3,
                       out_csv='/home/fabric/work/qkd-pqc-dependability/results/joint_fault_dist_loss_NCSA_KY.csv'):
    all_rows = []
    total = len(dist_values) * len(conditions) * n_runs
    done = 0
    write_header = not os.path.exists(out_csv)

    for dist in dist_values:
        scenario_content = f"""name: dist_{dist}
channel:
  distance_km: {dist}
  attenuation_db_per_km: 0.2
  polarization_fidelity: 0.98
detector:
  efficiency: 0.8
  dark_count_rate: 10.0
  dead_time: 0.0
  timing_jitter: 0.0
protocol:
  num_photons: 10000
  send_rate_hz: 10000.0
  sample_fraction: 0.1
  wavelength: 0
seed: 42
"""
        scenario_path = f'/home/fabric/work/qkd-pqc-dependability/validation/scenarios/temp_dist2_{dist}.yml'
        with open(scenario_path, 'w') as f:
            f.write(scenario_content)

        for run in range(n_runs):
            rows = deploy.run_network_conditions_experiment(slice_obj, scenario_path, conditions=conditions,
                                                               save_network_effects_json=False)
            for row in rows:
                row['distance_km'] = dist
                row['run'] = run + 1
                all_rows.append(row)
                with open(out_csv, 'a', newline='') as f:
                    writer = csv.DictWriter(f, fieldnames=FIELDNAMES_DIST2, extrasaction='ignore')
                    if write_header:
                        writer.writeheader()
                        write_header = False
                    writer.writerow(row)
            done += 1
            print(f"Progress: {done}/{total} (dist={dist}, run={run+1})")

    return all_rows

conditions = [{'name': 'baseline'}] + [{'name': f'loss_{l}pct', 'loss_pct': l} for l in [1, 5, 15]]
dist_values = [1, 10, 50, 100]

rows_dist2 = run_dist_loss_v2(slice_obj, dist_values, conditions, n_runs=3)


##### [1/4] classical condition: baseline #####
  cleared classical netem on Alice and Bob

=== Running BB84 protocol ===
  Bob data-plane IP: 10.10.1.2 (for classical channel)
  Alice data iface:  enp7s0
  Bob data iface:    enp7s0
  Cleaning up previous runs...
  Ensuring deps on alice...
  Ensuring deps on bob...
  Starting Bob...
  Alice MAC args:  --dst-mac '1A:9E:9B:43:74:D7' --src-mac '0E:A9:B5:58:70:AB'
  Starting Alice...
  Waiting for BB84 to complete...
  Alice finished
  Alice output: Alice: Sending 10000 photons on enp7s0
Alice: Finished sending 10000 photons
Alice: Connecting to Bob at 10.10.1.2:5100
  Retrying connection to 10.10.1.2:5100 (attempt 2/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 3/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 4/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 5/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 6/60)...
  Retrying connection to 10.10.1.2:5100 (attempt 7/60)...
  Retrying connection to 10.1

KeyboardInterrupt: 

In [ ]:
import os, csv
import deploy_fabric_modified as deploy

FIELDNAMES_ATTEN2 = ['condition', 'loss_pct', 'qber', 'sifted_bits', 'final_key_bits',
                       'secure_key_rate', 'elapsed_seconds', 'key_bits_per_sec',
                       'attenuation_db_per_km', 'run', 'note']

def run_atten_loss_v2(slice_obj, atten_values, conditions, n_runs=3,
                        out_csv='/home/fabric/work/qkd-pqc-dependability/results/joint_fault_atten_loss_NCSA_KY.csv'):
    all_rows = []
    total = len(atten_values) * len(conditions) * n_runs
    done = 0
    write_header = not os.path.exists(out_csv)

    for atten in atten_values:
        scenario_content = f"""name: atten_{atten}
channel:
  distance_km: 10.0
  attenuation_db_per_km: {atten}
  polarization_fidelity: 0.98
detector:
  efficiency: 0.8
  dark_count_rate: 10.0
  dead_time: 0.0
  timing_jitter: 0.0
protocol:
  num_photons: 10000
  send_rate_hz: 10000.0
  sample_fraction: 0.1
  wavelength: 0
seed: 42
"""
        scenario_path = f'/home/fabric/work/qkd-pqc-dependability/validation/scenarios/temp_atten2_{atten}.yml'
        with open(scenario_path, 'w') as f:
            f.write(scenario_content)

        for run in range(n_runs):
            rows = deploy.run_network_conditions_experiment(slice_obj, scenario_path, conditions=conditions,
                                                               save_network_effects_json=False)
            for row in rows:
                row['attenuation_db_per_km'] = atten
                row['run'] = run + 1
                all_rows.append(row)
                with open(out_csv, 'a', newline='') as f:
                    writer = csv.DictWriter(f, fieldnames=FIELDNAMES_ATTEN2, extrasaction='ignore')
                    if write_header:
                        writer.writeheader()
                        write_header = False
                    writer.writerow(row)
            done += 1
            print(f"Progress: {done}/{total} (atten={atten}, run={run+1})")

    return all_rows

conditions = [{'name': 'baseline'}] + [{'name': f'loss_{l}pct', 'loss_pct': l} for l in [1, 5, 15]]
atten_values = [0.1, 0.2, 0.4, 0.5]

rows_atten2 = run_atten_loss_v2(slice_obj, atten_values, conditions, n_runs=3)

In [ ]:
import pandas as pd
df_jitter = pd.read_csv('/home/fabric/work/qkd-pqc-dependability/results/joint_fault_jitter_check.csv')
print(df_jitter[['condition', 'delay_ms', 'jitter_ms', 'qber', 'sifted_bits', 'key_bits_per_sec']])